# Perseptron v2 - 04 Image-Only EfficientNet-B0 CNN

Bu notebook image-only CNN baseline modelini egitir. Bu model final personalized recommender degil; proposal icin gorsel baseline ve Grad-CAM kaynagi olarak kullanilir.

Ciktilar:

- `models/proposal_v2/image_only_effnet_cnn_fold{FOLD_ID}.pt`
- `reports/proposal_v2/proposal_v2_cnn_metrics.csv`

## Ortam ve output klasorleri

Bu hucre Kaggle icin yazilabilir output klasorlerini hazirlar ve onceki notebook outputlari `Add Data` ile eklendiyse bunlari geri yukler. Repo icindeki `.py` modulleri import edilmez.

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
REPORTS_DIR = WORK_DIR / 'reports' / 'proposal_v2'
MODELS_DIR = WORK_DIR / 'models' / 'proposal_v2'
GRADCAM_DIR = REPORTS_DIR / 'gradcam_examples'
for path in [REPORTS_DIR, REPORTS_DIR / 'folds', MODELS_DIR, GRADCAM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def candidate_output_roots(input_root):
    roots = []
    seen = set()

    def add_root(path):
        resolved = path.resolve()
        if resolved not in seen:
            roots.append(path)
            seen.add(resolved)

    notebooks_root = input_root / 'notebooks'
    if notebooks_root.exists():
        for path in notebooks_root.rglob('*'):
            if path.is_dir() and ((path / 'reports' / 'proposal_v2').exists() or (path / 'models' / 'proposal_v2').exists()):
                add_root(path)
    datasets_root = input_root / 'datasets'
    if datasets_root.exists():
        for path in datasets_root.rglob('*'):
            if not path.is_dir():
                continue
            name = path.name.lower()
            if any(token in name for token in ['proposal', 'report', 'output', 'fold']):
                add_root(path)
    for path in input_root.glob('*'):
        if path.is_dir() and path.name not in {'competitions', 'datasets', 'notebooks'}:
            add_root(path)
    return roots

def merge_or_copy_file(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.suffix == '.csv' and target.exists():
        try:
            current = pd.read_csv(target)
            incoming = pd.read_csv(source)
            merged = pd.concat([current, incoming], ignore_index=True)
            if {'fold_id', 'model'}.issubset(merged.columns):
                merged = merged.drop_duplicates(['fold_id', 'model'], keep='last')
            elif 'customer_id' in merged.columns and 'model' in merged.columns:
                keys = [column for column in ['fold_id', 'customer_id', 'model'] if column in merged.columns]
                merged = merged.drop_duplicates(keys, keep='last')
            else:
                merged = merged.drop_duplicates(keep='last')
            merged.to_csv(target, index=False)
            print(f'Merged previous CSV: {source} -> {target}')
            return
        except Exception as exc:
            print(f'CSV merge failed, falling back to copy for {source}: {exc}')
    if not target.exists() or source.stat().st_size != target.stat().st_size:
        shutil.copy2(source, target)
        print(f'Restored previous output file: {source} -> {target}')

def restore_previous_outputs():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return
    for root in candidate_output_roots(input_root):
        for source in root.rglob('*'):
            if not source.is_file():
                continue
            try:
                relative = source.relative_to(root)
            except ValueError:
                relative = Path(source.name)
            relative_text = relative.as_posix()
            name = source.name
            if relative_text.startswith('reports/proposal_v2/'):
                target = WORK_DIR / relative
            elif relative_text.startswith('models/proposal_v2/'):
                target = WORK_DIR / relative
            elif name.endswith('.pt'):
                target = MODELS_DIR / name
            elif name.endswith('_gradcam.png'):
                target = GRADCAM_DIR / name
            elif name.startswith('proposal_v2_') and name.endswith(('.csv', '.json', '.md')):
                target = REPORTS_DIR / name
            elif name.startswith('fold') and name.endswith('_context.json'):
                target = REPORTS_DIR / 'folds' / name
            else:
                continue
            merge_or_copy_file(source, target)

restore_previous_outputs()
print('WORK_DIR    =', WORK_DIR)
print('REPORTS_DIR =', REPORTS_DIR)
print('MODELS_DIR  =', MODELS_DIR)


## Ortak sabitler

Bu hucre pathleri, feature listelerini, model adlarini ve deney varsayilanlarini tanimlar.

In [ ]:
from dataclasses import dataclass

DATA_DIR = WORK_DIR / 'data'
REPORTS_DIR = WORK_DIR / 'reports' / 'proposal_v2'
MODELS_DIR = WORK_DIR / 'models' / 'proposal_v2'

DEFAULT_SEED = 42
DEFAULT_N_FOLDS = 5
DEFAULT_VALIDATION_DAYS = 7

CUSTOMER_NUMERIC_FEATURES = ['FN', 'Active', 'age']
ARTICLE_NUMERIC_FEATURES = [
    'product_code',
    'product_type_no',
    'graphical_appearance_no',
    'colour_group_code',
    'perceived_colour_value_id',
    'perceived_colour_master_id',
    'department_no',
    'index_group_no',
    'section_no',
    'garment_group_no',
]
VISUAL_NUMERIC_FEATURES = ['visual_similarity', 'visual_history_count']
TABULAR_NUMERIC_FEATURES = CUSTOMER_NUMERIC_FEATURES + ARTICLE_NUMERIC_FEATURES
FUSION_NUMERIC_FEATURES = TABULAR_NUMERIC_FEATURES + VISUAL_NUMERIC_FEATURES

CUSTOMER_CATEGORICAL_FEATURES = ['club_member_status', 'fashion_news_frequency']
ARTICLE_CATEGORICAL_FEATURES = [
    'product_type_name',
    'product_group_name',
    'graphical_appearance_name',
    'colour_group_name',
    'perceived_colour_value_name',
    'perceived_colour_master_name',
    'department_name',
    'index_code',
    'index_name',
    'index_group_name',
    'section_name',
    'garment_group_name',
]
CATEGORICAL_FEATURES = CUSTOMER_CATEGORICAL_FEATURES + ARTICLE_CATEGORICAL_FEATURES
MODEL_NAMES = ('tabular_only', 'image_history', 'late_fusion')

@dataclass(frozen=True)
class V2Defaults:
    n_folds: int = DEFAULT_N_FOLDS
    validation_days: int = DEFAULT_VALIDATION_DAYS
    seed: int = DEFAULT_SEED
    top_k: int = 12
    precision_k: int = 10
    candidate_limit: int = 5000
    visual_neighbors: int = 3000
    co_purchase_per_item: int = 300
    hybrid_weights: tuple[float, ...] = (0.25, 0.45, 0.65)
    negatives_per_positive: int = 1
    train_batch_size: int = 4096
    epochs: int = 3
    learning_rate: float = 1e-3

def ensure_v2_dirs():
    for path in [REPORTS_DIR, REPORTS_DIR / 'folds', REPORTS_DIR / 'gradcam_examples', MODELS_DIR]:
        path.mkdir(parents=True, exist_ok=True)

## Veri yukleme yardimcilari

Bu hucre H&M raw dosyalarini, image klasorunu ve EfficientNet embedding cache dosyalarini Kaggle inputlari veya local klasorlerden bulmak icin yardimci fonksiyonlari tanimlar.

In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd



def log(message: str) -> None:
    print(message, flush=True)


def _candidate_raw_dirs() -> list[Path]:
    candidates: list[Path] = []
    env_dir = os.environ.get("HM_RAW_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.append(DATA_DIR / "raw")
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for path in kaggle_root.rglob("transactions_train.csv"):
            candidates.append(path.parent)
    return candidates


def resolve_raw_dir(raw_dir: str | Path | None = None) -> Path:
    candidates = [Path(raw_dir)] if raw_dir else _candidate_raw_dirs()
    for candidate in candidates:
        if (candidate / "transactions_train.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find H&M raw data directory. Set HM_RAW_DIR or pass --raw-dir.")


def resolve_images_dir(raw_dir: Path, images_dir: str | Path | None = None) -> Path:
    if images_dir:
        return Path(images_dir)
    for candidate in [raw_dir / "images", DATA_DIR / "images" / "hm_images"]:
        if candidate.exists():
            return candidate
    return raw_dir / "images"


def _find_file_by_name(root: Path, name: str) -> Path | None:
    if not root.exists():
        return None
    for path in root.rglob(name):
        return path
    return None


def resolve_embedding_paths(
    embeddings_path: str | Path | None = None,
    embedding_ids_path: str | Path | None = None,
) -> tuple[Path, Path]:
    if embeddings_path and embedding_ids_path:
        return Path(embeddings_path), Path(embedding_ids_path)

    env_embeddings = os.environ.get("HM_EMBEDDINGS_PATH")
    env_ids = os.environ.get("HM_EMBEDDING_IDS_PATH")
    if env_embeddings and env_ids:
        return Path(env_embeddings), Path(env_ids)

    local_embeddings = DATA_DIR / "embeddings" / "final_kaggle" / "article_image_embeddings_popular.npy"
    local_ids = DATA_DIR / "embeddings" / "final_kaggle" / "article_image_embedding_ids_popular.csv"
    if local_embeddings.exists() and local_ids.exists():
        return local_embeddings, local_ids

    kaggle_root = Path("/kaggle/input")
    embeddings = _find_file_by_name(kaggle_root, "article_image_embeddings_popular.npy")
    ids = _find_file_by_name(kaggle_root, "article_image_embedding_ids_popular.csv")
    if embeddings and ids:
        return embeddings, ids

    raise FileNotFoundError("Could not find EfficientNet embedding cache paths.")


def read_transactions(raw_dir: Path) -> pd.DataFrame:
    transactions = pd.read_csv(raw_dir / "transactions_train.csv", dtype={"article_id": str})
    transactions["article_id"] = transactions["article_id"].astype(str).str.zfill(10)
    transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])
    transactions["price"] = transactions["price"].astype("float32")
    transactions["sales_channel_id"] = transactions["sales_channel_id"].astype("int8")
    return transactions


def read_customers(raw_dir: Path) -> pd.DataFrame:
    customers = pd.read_csv(raw_dir / "customers.csv")
    customers["FN"] = customers["FN"].fillna(0).astype("float32")
    customers["Active"] = customers["Active"].fillna(0).astype("float32")
    customers["age"] = customers["age"].fillna(customers["age"].median()).astype("float32")
    for column in ["club_member_status", "fashion_news_frequency"]:
        customers[column] = customers[column].fillna("UNKNOWN").astype(str)
    return customers


def read_articles(raw_dir: Path) -> pd.DataFrame:
    articles = pd.read_csv(raw_dir / "articles.csv", dtype={"article_id": str})
    articles["article_id"] = articles["article_id"].astype(str).str.zfill(10)
    for column in articles.columns:
        if articles[column].dtype == "object":
            articles[column] = articles[column].fillna("UNKNOWN").astype(str)
    return articles


def load_core_tables(raw_dir: str | Path | None = None) -> tuple[Path, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    resolved = resolve_raw_dir(raw_dir)
    log(f"Using raw data: {resolved}")
    return resolved, read_transactions(resolved), read_customers(resolved), read_articles(resolved)


def load_embeddings(
    embeddings_path: str | Path | None = None,
    embedding_ids_path: str | Path | None = None,
    mmap_mode: str | None = "r",
) -> tuple[np.ndarray, list[str], dict[str, int]]:
    emb_path, ids_path = resolve_embedding_paths(embeddings_path, embedding_ids_path)
    log(f"Using embeddings: {emb_path}")
    embeddings = np.load(emb_path, mmap_mode=mmap_mode)
    ids_frame = pd.read_csv(ids_path, dtype={"article_id": str})
    article_ids = ids_frame["article_id"].astype(str).str.zfill(10).tolist()
    article_to_index = {article_id: index for index, article_id in enumerate(article_ids)}
    return embeddings, article_ids, article_to_index


def article_image_path(images_dir: Path, article_id: str) -> Path:
    padded = str(article_id).zfill(10)
    nested = images_dir / padded[:3] / f"{padded}.jpg"
    if nested.exists():
        return nested
    return images_dir / f"{padded}.jpg"

## CNN yardimci kodu

Bu hucre image pair sampling, image dataset loading, EfficientNet-B0 classifier kurulumu ve evaluation yardimcilarini tanimlar.

In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd



def make_cutoff(transactions: pd.DataFrame, validation_days: int) -> pd.Timestamp:
    return transactions["t_dat"].max() - pd.Timedelta(days=validation_days)


def sample_positive_pairs(
    transactions: pd.DataFrame,
    customers: set[str],
    cutoff: pd.Timestamp,
    max_positives: int | None,
    seed: int,
) -> pd.DataFrame:
    positives = transactions[
        transactions["customer_id"].isin(customers) & (transactions["t_dat"] <= cutoff)
    ][["customer_id", "article_id"]].drop_duplicates()
    positives["label"] = 1
    if max_positives and len(positives) > max_positives:
        positives = positives.sample(max_positives, random_state=seed)
    return positives.reset_index(drop=True)


def make_validation_positive_pairs(
    transactions: pd.DataFrame,
    customers: set[str],
    cutoff: pd.Timestamp,
    max_positives: int | None,
    seed: int,
) -> pd.DataFrame:
    positives = transactions[
        transactions["customer_id"].isin(customers) & (transactions["t_dat"] > cutoff)
    ][["customer_id", "article_id"]].drop_duplicates()
    positives["label"] = 1
    if max_positives and len(positives) > max_positives:
        positives = positives.sample(max_positives, random_state=seed)
    return positives.reset_index(drop=True)


def add_negative_pairs(
    positives: pd.DataFrame,
    article_pool: np.ndarray,
    negatives_per_positive: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    neg_customers = np.repeat(positives["customer_id"].to_numpy(), negatives_per_positive)
    neg_articles = rng.choice(article_pool, size=len(neg_customers), replace=True)
    negatives = pd.DataFrame({"customer_id": neg_customers, "article_id": neg_articles, "label": 0})
    data = pd.concat([positives, negatives], ignore_index=True)
    data = data.drop_duplicates(["customer_id", "article_id", "label"])
    return data.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def build_history_frame(transactions: pd.DataFrame, customers: set[str], cutoff: pd.Timestamp) -> pd.DataFrame:
    return transactions[
        transactions["customer_id"].isin(customers) & (transactions["t_dat"] <= cutoff)
    ][["customer_id", "article_id"]]


def build_customer_profiles(
    history: pd.DataFrame,
    embeddings: np.ndarray,
    article_to_index: dict[str, int],
) -> tuple[dict[str, int], np.ndarray, np.ndarray, dict[tuple[str, str], int]]:
    history = history[history["article_id"].isin(article_to_index)].copy()
    customer_ids = sorted(history["customer_id"].unique())
    customer_to_index = {customer_id: index for index, customer_id in enumerate(customer_ids)}
    sums = np.zeros((len(customer_ids), embeddings.shape[1]), dtype="float32")
    counts = np.zeros(len(customer_ids), dtype="float32")
    pair_counts: dict[tuple[str, str], int] = {}
    for row in history.itertuples(index=False):
        customer_idx = customer_to_index[row.customer_id]
        article_idx = article_to_index[row.article_id]
        sums[customer_idx] += embeddings[article_idx]
        counts[customer_idx] += 1.0
        key = (row.customer_id, row.article_id)
        pair_counts[key] = pair_counts.get(key, 0) + 1
    return customer_to_index, sums, counts, pair_counts


def l2_normalize_rows(values: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, eps)


def visual_arrays_for_pairs(
    pairs: pd.DataFrame,
    embeddings: np.ndarray,
    article_to_index: dict[str, int],
    customer_to_index: dict[str, int],
    profile_sums: np.ndarray,
    profile_counts: np.ndarray,
    pair_counts: dict[tuple[str, str], int],
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    image_dim = embeddings.shape[1]
    article_emb = np.zeros((len(pairs), image_dim), dtype="float32")
    profile_emb = np.zeros((len(pairs), image_dim), dtype="float32")
    visual_similarity = np.zeros(len(pairs), dtype="float32")
    visual_history_count = np.zeros(len(pairs), dtype="float32")

    for index, row in enumerate(pairs[["customer_id", "article_id"]].itertuples(index=False)):
        article_idx = article_to_index.get(row.article_id)
        customer_idx = customer_to_index.get(row.customer_id)
        if article_idx is None or customer_idx is None:
            continue
        candidate = np.asarray(embeddings[article_idx], dtype="float32")
        count_to_remove = pair_counts.get((row.customer_id, row.article_id), 0)
        usable_count = max(float(profile_counts[customer_idx] - count_to_remove), 0.0)
        if usable_count > 0:
            profile = (profile_sums[customer_idx] - count_to_remove * candidate) / usable_count
        else:
            profile = np.zeros(image_dim, dtype="float32")
        article_emb[index] = candidate
        profile_emb[index] = profile
        visual_history_count[index] = usable_count
        denom = np.linalg.norm(candidate) * np.linalg.norm(profile)
        visual_similarity[index] = float(np.dot(candidate, profile) / denom) if denom > 0 else 0.0
    return article_emb, profile_emb, visual_similarity, visual_history_count


def merge_metadata(pairs: pd.DataFrame, customers: pd.DataFrame, articles: pd.DataFrame) -> pd.DataFrame:
    frame = pairs.merge(customers, on="customer_id", how="left").merge(articles, on="article_id", how="left")
    return frame


def fit_tabular_metadata(frame: pd.DataFrame, numeric_features: list[str]) -> dict:
    metadata = {
        "numeric_features": numeric_features,
        "categorical_features": CATEGORICAL_FEATURES,
        "numeric_mean": {},
        "numeric_std": {},
        "category_maps": {},
    }
    for column in numeric_features:
        values = pd.to_numeric(frame[column], errors="coerce").astype("float32")
        metadata["numeric_mean"][column] = float(values.mean()) if len(values) else 0.0
        std = float(values.std()) if len(values) else 1.0
        metadata["numeric_std"][column] = std if std > 1e-8 else 1.0
    for column in CATEGORICAL_FEATURES:
        values = frame[column].fillna("UNKNOWN").astype(str)
        categories = ["__UNK__"] + sorted(values.unique().tolist())
        metadata["category_maps"][column] = {value: index for index, value in enumerate(categories)}
    return metadata


def encode_tabular(frame: pd.DataFrame, metadata: dict) -> tuple[np.ndarray, np.ndarray]:
    numeric_columns = []
    for column in metadata["numeric_features"]:
        values = pd.to_numeric(frame[column], errors="coerce").fillna(metadata["numeric_mean"][column]).astype("float32")
        values = (values - metadata["numeric_mean"][column]) / metadata["numeric_std"][column]
        numeric_columns.append(values.to_numpy(dtype="float32"))
    numeric = np.stack(numeric_columns, axis=1).astype("float32")

    categorical_columns = []
    for column in metadata["categorical_features"]:
        mapping = metadata["category_maps"][column]
        values = frame[column].fillna("UNKNOWN").astype(str).map(mapping).fillna(0).astype("int64")
        categorical_columns.append(values.to_numpy(dtype="int64"))
    categorical = np.stack(categorical_columns, axis=1).astype("int64")
    return numeric, categorical


def category_sizes(metadata: dict) -> list[int]:
    return [len(metadata["category_maps"][column]) for column in metadata["categorical_features"]]


def metadata_to_jsonable(metadata: dict) -> dict:
    return json.loads(json.dumps(metadata))


def numeric_features_for_model(model_name: str) -> list[str]:
    if model_name == "late_fusion":
        return FUSION_NUMERIC_FEATURES
    return TABULAR_NUMERIC_FEATURES



import torch
from torch import nn
from torchvision import models


def embedding_dim(size: int) -> int:
    return min(50, max(4, int(size**0.25 * 8)))


class TabularOnlyMLP(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int]) -> None:
        super().__init__()
        self.embeddings = nn.ModuleList(
            [nn.Embedding(size, embedding_dim(size)) for size in category_sizes]
        )
        cat_dim = sum(embedding.embedding_dim for embedding in self.embeddings)
        self.net = nn.Sequential(
            nn.Linear(numeric_dim + cat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, numeric: torch.Tensor, categorical: torch.Tensor) -> torch.Tensor:
        embedded = [emb(categorical[:, idx]) for idx, emb in enumerate(self.embeddings)]
        x = torch.cat([numeric, *embedded], dim=1)
        return self.net(x).squeeze(1)


class ImageHistoryMLP(nn.Module):
    def __init__(self, image_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(
        self,
        article_embedding: torch.Tensor,
        profile_embedding: torch.Tensor,
        visual_similarity: torch.Tensor,
        visual_history_count: torch.Tensor,
    ) -> torch.Tensor:
        visual_extra = torch.stack([visual_similarity, visual_history_count], dim=1)
        x = torch.cat([article_embedding, profile_embedding, visual_extra], dim=1)
        return self.net(x).squeeze(1)


class MultimodalLateFusion(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int], image_dim: int) -> None:
        super().__init__()
        self.embeddings = nn.ModuleList(
            [nn.Embedding(size, embedding_dim(size)) for size in category_sizes]
        )
        cat_dim = sum(embedding.embedding_dim for embedding in self.embeddings)
        self.tabular_branch = nn.Sequential(
            nn.Linear(numeric_dim + cat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.visual_branch = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 128),
            nn.ReLU(),
        )
        self.fusion_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(
        self,
        numeric: torch.Tensor,
        categorical: torch.Tensor,
        article_embedding: torch.Tensor,
        profile_embedding: torch.Tensor,
        visual_similarity: torch.Tensor,
        visual_history_count: torch.Tensor,
    ) -> torch.Tensor:
        embedded = [emb(categorical[:, idx]) for idx, emb in enumerate(self.embeddings)]
        tabular = self.tabular_branch(torch.cat([numeric, *embedded], dim=1))
        visual_extra = torch.stack([visual_similarity, visual_history_count], dim=1)
        visual = self.visual_branch(torch.cat([article_embedding, profile_embedding, visual_extra], dim=1))
        return self.fusion_head(torch.cat([tabular, visual], dim=1)).squeeze(1)


class EfficientNetBinaryClassifier(nn.Module):
    def __init__(self, train_backbone: bool = False) -> None:
        super().__init__()
        try:
            weights = models.EfficientNet_B0_Weights.DEFAULT
            self.weights = weights
            self.backbone = models.efficientnet_b0(weights=weights)
        except Exception as exc:
            print('Pretrained EfficientNet weights unavailable, using random weights:', exc)
            weights = None
            self.weights = weights
            self.backbone = models.efficientnet_b0(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, 1))
        if not train_backbone:
            for parameter in self.backbone.features.parameters():
                parameter.requires_grad = False

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.backbone(images).squeeze(1)


def build_model(model_name: str, metadata: dict, image_dim: int) -> nn.Module:
    category_sizes = [len(metadata["category_maps"][column]) for column in metadata["categorical_features"]]
    numeric_dim = len(metadata["numeric_features"])
    if model_name == "tabular_only":
        return TabularOnlyMLP(numeric_dim, category_sizes)
    if model_name == "image_history":
        return ImageHistoryMLP(image_dim)
    if model_name == "late_fusion":
        return MultimodalLateFusion(numeric_dim, category_sizes, image_dim)
    raise ValueError(f"Unknown model: {model_name}")



import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm



class ArticleImagePairDataset(Dataset):
    def __init__(self, pairs: pd.DataFrame, images_dir: Path, transform) -> None:
        self.pairs = pairs.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int):
        row = self.pairs.iloc[index]
        path = article_image_path(self.images_dir, row["article_id"])
        image = Image.open(path).convert("RGB")
        return self.transform(image), torch.tensor(float(row["label"]), dtype=torch.float32)


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            probs = torch.sigmoid(logits).cpu().numpy()
            y_prob.extend(probs.tolist())
            y_true.extend(labels.numpy().tolist())
    labels = np.array(y_true)
    probs = np.array(y_prob)
    return {
        "auc_roc": float(roc_auc_score(labels, probs)) if len(np.unique(labels)) > 1 else float("nan"),
        "accuracy": float(accuracy_score(labels, probs >= 0.5)),
    }


def main() -> None:
    parser = argparse.ArgumentParser(description="Train proposal v2 EfficientNet-B0 image-only CNN baseline.")
    parser.add_argument("--raw-dir", type=Path, default=None)
    parser.add_argument("--images-dir", type=Path, default=None)
    parser.add_argument("--folds-csv", type=Path, default=REPORTS_DIR / "proposal_v2_fold_splits.csv")
    parser.add_argument("--fold-id", type=int, default=0)
    parser.add_argument("--validation-days", type=int, default=7)
    parser.add_argument("--max-train-positives", type=int, default=20_000)
    parser.add_argument("--max-val-positives", type=int, default=5_000)
    parser.add_argument("--negatives-per-positive", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--train-backbone", action="store_true")
    parser.add_argument("--seed", type=int, default=DEFAULT_SEED)
    parser.add_argument("--device", default=None)
    parser.add_argument("--model-dir", type=Path, default=MODELS_DIR)
    parser.add_argument("--metrics-csv", type=Path, default=REPORTS_DIR / "proposal_v2_cnn_metrics.csv")
    args = parser.parse_args()

    ensure_v2_dirs()
    torch.manual_seed(args.seed)
    raw_dir, transactions, _, articles = load_core_tables(args.raw_dir)
    images_dir = resolve_images_dir(raw_dir, args.images_dir)
    folds = pd.read_csv(args.folds_csv)
    cutoff = make_cutoff(transactions, args.validation_days)
    val_customers = set(folds.loc[folds["fold_id"] == args.fold_id, "customer_id"])
    train_customers = set(folds.loc[folds["fold_id"] != args.fold_id, "customer_id"])
    article_pool = articles["article_id"].astype(str).to_numpy()

    train_positive = sample_positive_pairs(transactions, train_customers, cutoff, args.max_train_positives, args.seed)
    val_positive = make_validation_positive_pairs(transactions, val_customers, cutoff, args.max_val_positives, args.seed)
    train_pairs = add_negative_pairs(train_positive, article_pool, args.negatives_per_positive, args.seed)
    val_pairs = add_negative_pairs(val_positive, article_pool, args.negatives_per_positive, args.seed + 1000)
    train_pairs = train_pairs[train_pairs["article_id"].map(lambda value: article_image_path(images_dir, value).exists())]
    val_pairs = val_pairs[val_pairs["article_id"].map(lambda value: article_image_path(images_dir, value).exists())]

    model = EfficientNetBinaryClassifier(train_backbone=args.train_backbone)
    if model.weights is not None:
        transform = model.weights.transforms()
    else:
        from torchvision import transforms
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    train_loader = DataLoader(ArticleImagePairDataset(train_pairs, images_dir, transform), batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(ArticleImagePairDataset(val_pairs, images_dir, transform), batch_size=args.batch_size, shuffle=False)
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model.to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.learning_rate)
    criterion = nn.BCEWithLogitsLoss()

    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for images, labels in tqdm(train_loader, desc=f"image_only_effnet_cnn epoch {epoch}/{args.epochs}"):
            optimizer.zero_grad(set_to_none=True)
            logits = model(images.to(device))
            loss = criterion(logits, labels.to(device))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        metrics = evaluate(model, val_loader, device)
        metrics.update({"epoch": epoch, "loss": float(np.mean(losses)) if losses else float("nan")})
        history.append(metrics)
        log(f"cnn epoch {epoch}: loss={metrics['loss']:.4f} auc={metrics['auc_roc']:.4f} acc={metrics['accuracy']:.4f}")

    args.model_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = args.model_dir / f"image_only_effnet_cnn_fold{args.fold_id}.pt"
    torch.save(
        {
            "model_name": "image_only_effnet_cnn",
            "fold_id": args.fold_id,
            "state_dict": model.state_dict(),
            "train_backbone": args.train_backbone,
            "metrics": {"history": history, "final": history[-1] if history else {}},
        },
        checkpoint_path,
    )
    row = {
        "fold_id": args.fold_id,
        "model": "image_only_effnet_cnn",
        "auc_roc": history[-1]["auc_roc"],
        "accuracy": history[-1]["accuracy"],
        "train_rows": len(train_pairs),
        "validation_rows": len(val_pairs),
        "checkpoint": str(checkpoint_path),
    }
    output = pd.DataFrame([row])
    if args.metrics_csv.exists():
        previous = pd.read_csv(args.metrics_csv)
        output = pd.concat([previous, output], ignore_index=True).drop_duplicates(["fold_id", "model"], keep="last")
    output.to_csv(args.metrics_csv, index=False)
    log(f"Saved CNN checkpoint: {checkpoint_path}")
    log(f"Saved CNN metrics: {args.metrics_csv}")

## Parametreler

Once `FAST_RUN=True` ile kisa smoke kosusu yapilir. Final temsilci egitim icin `FAST_RUN=False` kullanilir; sure kisitliysa fold 0 yeterlidir.

In [ ]:
FAST_RUN = True
FOLD_ID = 0
RAW_DIR = None
IMAGES_DIR = None
FOLDS_CSV = REPORTS_DIR / 'proposal_v2_fold_splits.csv'
VALIDATION_DAYS = 7
MAX_TRAIN_POSITIVES = 200 if FAST_RUN else 20_000
MAX_VAL_POSITIVES = 80 if FAST_RUN else 5_000
NEGATIVES_PER_POSITIVE = 1
EPOCHS = 1 if FAST_RUN else 3
BATCH_SIZE = 32 if FAST_RUN else 64
LEARNING_RATE = 1e-3
TRAIN_BACKBONE = False
RANDOM_SEED = 42
DEVICE = None

## CNN baseline egitimi

Bu hucre CNN egitim akisini notebook degiskenleriyle calistirir, checkpoint dosyasini kaydeder ve CNN metrics CSV dosyasini yazar.

In [ ]:
ensure_v2_dirs()
torch.manual_seed(RANDOM_SEED)
args = argparse.Namespace(
    raw_dir=RAW_DIR,
    images_dir=IMAGES_DIR,
    folds_csv=FOLDS_CSV,
    fold_id=FOLD_ID,
    validation_days=VALIDATION_DAYS,
    max_train_positives=MAX_TRAIN_POSITIVES,
    max_val_positives=MAX_VAL_POSITIVES,
    negatives_per_positive=NEGATIVES_PER_POSITIVE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    train_backbone=TRAIN_BACKBONE,
    seed=RANDOM_SEED,
    device=DEVICE,
    model_dir=MODELS_DIR,
    metrics_csv=REPORTS_DIR / 'proposal_v2_cnn_metrics.csv',
)
raw_dir, transactions, _, articles = load_core_tables(args.raw_dir)
images_dir = resolve_images_dir(raw_dir, args.images_dir)
folds = pd.read_csv(args.folds_csv)
cutoff = make_cutoff(transactions, args.validation_days)
val_customers = set(folds.loc[folds['fold_id'] == args.fold_id, 'customer_id'])
train_customers = set(folds.loc[folds['fold_id'] != args.fold_id, 'customer_id'])
article_pool = articles['article_id'].astype(str).to_numpy()
train_positive = sample_positive_pairs(transactions, train_customers, cutoff, args.max_train_positives, args.seed)
val_positive = make_validation_positive_pairs(transactions, val_customers, cutoff, args.max_val_positives, args.seed)
train_pairs = add_negative_pairs(train_positive, article_pool, args.negatives_per_positive, args.seed)
val_pairs = add_negative_pairs(val_positive, article_pool, args.negatives_per_positive, args.seed + 1000)
train_pairs = train_pairs[train_pairs['article_id'].map(lambda value: article_image_path(images_dir, value).exists())]
val_pairs = val_pairs[val_pairs['article_id'].map(lambda value: article_image_path(images_dir, value).exists())]

model = EfficientNetBinaryClassifier(train_backbone=args.train_backbone)
if model.weights is not None:
    transform = model.weights.transforms()
else:
    from torchvision import transforms
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
train_loader = DataLoader(ArticleImagePairDataset(train_pairs, images_dir, transform), batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(ArticleImagePairDataset(val_pairs, images_dir, transform), batch_size=args.batch_size, shuffle=False)
device = torch.device(args.device or ('cuda' if torch.cuda.is_available() else 'cpu'))
model.to(device)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.learning_rate)
criterion = nn.BCEWithLogitsLoss()
history = []
for epoch in range(1, args.epochs + 1):
    model.train()
    losses = []
    for images, labels in tqdm(train_loader, desc=f'image_only_effnet_cnn epoch {epoch}/{args.epochs}'):
        optimizer.zero_grad(set_to_none=True)
        logits = model(images.to(device))
        loss = criterion(logits, labels.to(device))
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    metrics = evaluate(model, val_loader, device)
    metrics.update({'epoch': epoch, 'loss': float(np.mean(losses)) if losses else float('nan')})
    history.append(metrics)
    log(f"cnn epoch {epoch}: loss={metrics['loss']:.4f} auc={metrics['auc_roc']:.4f} acc={metrics['accuracy']:.4f}")

checkpoint_path = args.model_dir / f'image_only_effnet_cnn_fold{args.fold_id}.pt'
torch.save({'model_name': 'image_only_effnet_cnn', 'fold_id': args.fold_id, 'state_dict': model.state_dict(), 'train_backbone': args.train_backbone, 'metrics': {'history': history, 'final': history[-1] if history else {}}}, checkpoint_path)
row = {'fold_id': args.fold_id, 'model': 'image_only_effnet_cnn', 'auc_roc': history[-1]['auc_roc'], 'accuracy': history[-1]['accuracy'], 'train_rows': len(train_pairs), 'validation_rows': len(val_pairs), 'checkpoint': str(checkpoint_path)}
output = pd.DataFrame([row])
if args.metrics_csv.exists():
    previous = pd.read_csv(args.metrics_csv)
    output = pd.concat([previous, output], ignore_index=True).drop_duplicates(['fold_id', 'model'], keep='last')
output.to_csv(args.metrics_csv, index=False)
print('Saved CNN checkpoint:', checkpoint_path)
print('Saved CNN metrics:', args.metrics_csv)
display(output.tail())